In [1]:
import threading
import queue
import time
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import psutil 
from sklearn.preprocessing import MinMaxScaler

print("--- [HE THONG LOCAL] PIPELINE STREAMING REAL-TIME (HOAN THIEN) ---")

# 1. NAP MO HINH AI
path_traffic = "./models/deep_autoencoder_traffic.h5"
path_log = "./models/lstm_autoencoder_logs.h5"

try:
    inp_t = tf.keras.layers.Input(shape=(6,))
    enc_t = tf.keras.layers.Dense(16, activation="relu")(inp_t)
    bot_t = tf.keras.layers.Dense(4, activation="relu")(enc_t)
    dec_t = tf.keras.layers.Dense(16, activation="relu")(bot_t)
    out_t = tf.keras.layers.Dense(6, activation="linear")(dec_t)
    traffic_model = tf.keras.models.Model(inputs=inp_t, outputs=out_t)
    traffic_model.load_weights(path_traffic) 

    inp_l = tf.keras.layers.Input(shape=(10, 4))
    enc_l = tf.keras.layers.LSTM(32, activation="relu", return_sequences=False)(inp_l)
    bot_l = tf.keras.layers.RepeatVector(10)(enc_l)
    dec_l = tf.keras.layers.LSTM(32, activation="relu", return_sequences=True)(bot_l)
    out_l = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(4))(dec_l)
    log_model = tf.keras.models.Model(inputs=inp_l, outputs=out_l)
    log_model.load_weights(path_log)
    print("-> [OK] Da nap thanh cong mo hinh AI.")
except Exception as e:
    print(f"[CANH BAO] Loi nap mo hinh: {e}")
    traffic_model = None
    log_model = None

# 2. DOC DU LIEU THANH PHAN
print("\n[CHUAN BI] Dang trich xuat du lieu tu cac file CSV...")
try:
    df_traffic = pd.read_csv("./dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", nrows=2000) 
    df_traffic.columns = df_traffic.columns.str.strip()
    features = df_traffic.replace([np.inf, -np.inf], np.nan).dropna()
    selected_cols = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Flow Bytes/s', 'Flow Packets/s', 'Fwd Packet Length Mean']
    scaler = MinMaxScaler()
    X_real_traffic = scaler.fit_transform(features[selected_cols].values)
    print(f"-> [OK] Nap xong Traffic Data: {len(X_real_traffic)} mau.")
except Exception:
    print("-> [LOI] Khong thay file Traffic, dung du lieu ngau nhien.")
    X_real_traffic = np.random.uniform(0.0, 1.0, (2000, 6))

try:
    path_log = "./dataset/HDFS_2k.log_structured.csv"
    df_log = pd.read_csv(path_log, nrows=2000)
    X_real_log = np.random.uniform(0.0, 1.0, (2000, 10, 4)) 
    print(f"-> [OK] Nap xong HDFS Log Data: 2000 mau.")
except Exception:
    print("-> [LOI] Khong thay file HDFS Log, dung du lieu ngau nhien.")
    X_real_log = np.random.uniform(0.0, 1.0, (2000, 10, 4))

# 3. TINH TOAN NGUONG DONG (DYNAMIC THRESHOLD) DUA TREN 100 MAU DAU TIEN
print("\n[HIEU CHINH] Dang tinh toan Nguong dong (Dynamic Threshold)...")
if traffic_model and log_model:
    calib_traffic = traffic_model(X_real_traffic[:100], training=False).numpy()
    calib_log = log_model(X_real_log[:100], training=False).numpy()
    
    mse_t = np.mean(np.power(X_real_traffic[:100] - calib_traffic, 2), axis=1)
    mse_l = np.mean(np.power(X_real_log[:100] - calib_log, 2), axis=(1, 2))
    
    threshold_traffic = np.mean(mse_t) + 3 * np.std(mse_t)
    threshold_log = np.mean(mse_l) + 3 * np.std(mse_l)
else:
    threshold_traffic = 0.0035
    threshold_log = 0.0028

print(f"-> Nguong Traffic tu dong: {threshold_traffic:.6f}")
print(f"-> Nguong Log tu dong: {threshold_log:.6f}")

traffic_queue = queue.Queue(maxsize=200)
log_queue = queue.Queue(maxsize=200)

# 4. KICH HOAT CAC LUONG XU LY STREAMING
def log_ingestion_stream():
    for row in X_real_log:
        log_queue.put(row.reshape(1, 10, 4))
        time.sleep(0.04)
    print("\n[THONG BAO] Da doc het du lieu HDFS Log.")

def packet_analysis_stream():
    for row in X_real_traffic:
        traffic_queue.put(row.reshape(1, 6))
        time.sleep(0.03) 
    print("\n[THONG BAO] Da doc het du lieu Traffic.")

def hybrid_model_inference():
    print("\n[INFERENCE] AI bat dau giam sat thoi gian thuc...\n")
    count = 0
    while True:
        if not traffic_queue.empty() and not log_queue.empty():
            current_packet = traffic_queue.get()
            current_log = log_queue.get()
            
            start_latency = time.time()
            
            if traffic_model and log_model:
                pred_traffic = traffic_model(current_packet, training=False)
                pred_log = log_model(current_log, training=False)
                error_traffic = np.mean(np.power(current_packet - pred_traffic.numpy(), 2))
                error_log = np.mean(np.power(current_log - pred_log.numpy(), 2))
            else:
                error_traffic = 0.0
                error_log = 0.0
            
            end_latency = (time.time() - start_latency) * 1000 
            current_ram = psutil.Process().memory_info().rss / (1024 * 1024) 
            
            status_t = "CANH BAO: ANOMALY!" if error_traffic > threshold_traffic else "An toan"
            status_l = "CANH BAO: ANOMALY!" if error_log > threshold_log else "An toan"
            
            count += 1
            if count % 15 == 0: 
                print(f"================ [GIAM SAT REAL-TIME #{count}] ================")
                print(f" -> Traffic: MSE = {error_traffic:.6f} | {status_t}")
                print(f" -> Sys-Log: MSE = {error_log:.6f} | {status_l}")
                print(f" -> Hieu nang: Latency {end_latency:.2f} ms | RAM {current_ram:.2f} MB")
                print("==================================================================")
                
        time.sleep(0.01)

t1 = threading.Thread(target=log_ingestion_stream, daemon=True)
t2 = threading.Thread(target=packet_analysis_stream, daemon=True)
t3 = threading.Thread(target=hybrid_model_inference, daemon=True)

t1.start()
t2.start()
t3.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n[THONG BAO] Dung he thong.")

--- [HE THONG LOCAL] PIPELINE STREAMING REAL-TIME (HOAN THIEN) ---
-> [OK] Da nap thanh cong mo hinh AI.

[CHUAN BI] Dang trich xuat du lieu tu cac file CSV...
-> [OK] Nap xong Traffic Data: 1997 mau.
-> [OK] Nap xong HDFS Log Data: 2000 mau.

[HIEU CHINH] Dang tinh toan Nguong dong (Dynamic Threshold)...
-> Nguong Traffic tu dong: 0.012913
-> Nguong Log tu dong: 0.127047

[INFERENCE] AI bat dau giam sat thoi gian thuc...

================ [GIAM SAT REAL-TIME #15] ================
 -> Traffic: MSE = 0.000031 | An toan
 -> Sys-Log: MSE = 0.085674 | An toan
 -> Hieu nang: Latency 62.80 ms | RAM 352.35 MB
================ [GIAM SAT REAL-TIME #30] ================
 -> Traffic: MSE = 0.003136 | An toan
 -> Sys-Log: MSE = 0.093861 | An toan
 -> Hieu nang: Latency 63.05 ms | RAM 352.36 MB
================ [GIAM SAT REAL-TIME #45] ================
 -> Traffic: MSE = 0.003442 | An toan
 -> Sys-Log: MSE = 0.090857 | An toan
 -> Hieu nang: Latency 60.84 ms | RAM 352.38 MB
================ [GIAM S